In [3]:
import os
import re
import glob
import torch
import numpy as np
from collections import defaultdict
import pandas as pd
import sys

# For round_to_perm
sys.path.append(os.path.abspath(os.path.join('..', 'src')))
from utils import round_to_perm

# ------------------ CLI ------------------
# Usage:
#   python summarize_varyB.py B n_i
#   python summarize_varyB.py B n_i phi snr
# if len(sys.argv) < 3:
#     raise SystemExit("Usage: python summarize_varyB.py B n_i [phi snr]")

B = 144
n_i = 6

phi = None
snr = None
# if len(sys.argv) >= 5:
#     phi = float(sys.argv[3])
#     snr = float(sys.argv[4])

# ------------------ Paths & autodetect ------------------
base_results_dir = os.path.join('..', 'data', 'results_varyB', 'results', 'vary_B', f'B_{B}_n_{n_i}')
base_data_dir    = os.path.join('..', 'data', 'vary_B', 'B_{}_n_{}'.format(B, n_i))

def autodetect_phi_snr_dir(base_dir):
    if not os.path.isdir(base_dir):
        raise FileNotFoundError(f"Folder not found: {base_dir}")
    cands = [d for d in os.listdir(base_dir) if os.path.isdir(os.path.join(base_dir, d)) and d.startswith('phi_')]
    if len(cands) == 0:
        raise FileNotFoundError(f"No phi/snr subfolders under {base_dir}")
    if len(cands) > 1:
        raise ValueError(f"Multiple phi/snr folders under {base_dir}: {cands}. "
                         f"Re-run with explicit phi and snr.")
    return cands[0]  # e.g., 'phi_2.0_snr_1.0e+00'

if phi is None or snr is None:
    phi_snr_dir = autodetect_phi_snr_dir(base_results_dir)
else:
    phi_snr_dir = f"phi_{phi}_snr_{snr:.1e}"

result_folder = os.path.join(base_results_dir, phi_snr_dir)
data_folder   = os.path.join(base_data_dir,   phi_snr_dir)

# ------------------ Load a data file to get true perms ------------------
# Use any seed file present
data_candidates = sorted(glob.glob(os.path.join(data_folder, 'data_seed_*.pt')))
if len(data_candidates) == 0:
    raise FileNotFoundError(f"No data files found at {data_folder}/data_seed_*.pt")
data = torch.load(data_candidates[0], map_location='cpu')

perm_matrix_x = data['perm_matrix_x']
perm_matrix_s = data['perm_matrix_s']

# ------------------ Aggregation structs ------------------
def summarize(values):
    return {'mean': np.nanmean(values), 'sd': np.nanstd(values)}

stats = {
    'GPmodel': defaultdict(list),
    'GPArealModel': defaultdict(list),
    'VIGP_unlinked': defaultdict(lambda: defaultdict(list))
}

# ------------------ Sweep result files ------------------
result_files = sorted(glob.glob(os.path.join(result_folder, 'results_seed_*.pt')))
if len(result_files) == 0:
    raise FileNotFoundError(f"No result files found at {result_folder}/results_seed_*.pt")

all_tau_values = set()

for path in result_files:
    # Load the result file
    result = torch.load(path, map_location='cpu', weights_only=False)

    # GP & Areal summaries
    for model in ['GPmodel', 'GPArealModel']:
        if model in result:
            m = result[model]
            beta_val = m.get('beta', [np.nan])
            if isinstance(beta_val, torch.Tensor):
                beta_val = beta_val.detach().cpu().numpy()
            # assume 1D beta
            stats[model]['beta'].append(beta_val[0] if np.ndim(beta_val) else beta_val)

            for key in ['sigmasq', 'tausq', 'phi']:
                val = m.get(key, np.nan)
                if isinstance(val, torch.Tensor):
                    val = val.item()
                stats[model][key].append(val)

            # placeholders (not applicable) to keep table shape
            stats[model]['n_correct_perm_x'].append(np.nan)
            stats[model]['n_correct_perm_s'].append(np.nan)

    # VI summaries per tau
    tau_keys = sorted(float(k.split('_')[-1]) for k in result if k.startswith('VIGP_unlinked_tau_'))
    for tau in tau_keys:
        key = f'VIGP_unlinked_tau_{tau}'
        vi = result.get(key, {})
        all_tau_values.add(tau)

        def as_float(x):
            if isinstance(x, torch.Tensor):
                x = x.detach().cpu().numpy()
            return x

        beta = as_float(vi.get('mu_lambda_beta', np.nan))
        sig_beta = as_float(vi.get('sigmasq_lambda_beta', np.nan))
        a1, b1 = as_float(vi.get('lambda_a1', np.nan)), as_float(vi.get('lambda_b1', np.nan))
        a2, b2 = as_float(vi.get('lambda_a2', np.nan)), as_float(vi.get('lambda_b2', np.nan))

        # Means (Gamma moments if shape>1)
        sigmasq = b1 / (a1 - 1) if (isinstance(a1, (int, float, np.floating)) and a1 > 1) else np.nan
        tausq   = b2 / (a2 - 1) if (isinstance(a2, (int, float, np.floating)) and a2 > 1) else np.nan

        # SDs
        sd_sigmasq = (b1**2) / ((a1 - 1)**2 * (a1 - 2)) if (isinstance(a1, (int, float, np.floating)) and a1 > 2) else np.nan
        sd_tausq   = (b2**2) / ((a2 - 1)**2 * (a2 - 2)) if (isinstance(a2, (int, float, np.floating)) and a2 > 2) else np.nan

        phi_mean = as_float(vi.get('mean_phi', np.nan))

        stats['VIGP_unlinked'][tau]['beta'].append(beta)
        stats['VIGP_unlinked'][tau]['sigmasq'].append(sigmasq)
        stats['VIGP_unlinked'][tau]['tausq'].append(tausq)
        stats['VIGP_unlinked'][tau]['phi'].append(phi_mean)
        stats['VIGP_unlinked'][tau]['sd_sigmasq'].append(sd_sigmasq)
        stats['VIGP_unlinked'][tau]['sd_tausq'].append(sd_tausq)
        stats['VIGP_unlinked'][tau]['sd_beta'].append(np.sqrt(sig_beta))

        # Perm accuracy
        M_X_star = as_float(vi.get('M_X_star', np.nan))
        M_S_star = as_float(vi.get('M_S_star', np.nan))
        if isinstance(M_X_star, np.ndarray) and isinstance(M_S_star, np.ndarray):
            n_correct_perm_x = torch.sum(perm_matrix_x.T * torch.from_numpy(round_to_perm(M_X_star))).item()
            n_correct_perm_s = torch.sum(perm_matrix_s.T * torch.from_numpy(round_to_perm(M_S_star))).item()
        else:
            n_correct_perm_x = np.nan
            n_correct_perm_s = np.nan

        stats['VIGP_unlinked'][tau]['n_correct_perm_x'].append(n_correct_perm_x)
        stats['VIGP_unlinked'][tau]['n_correct_perm_s'].append(n_correct_perm_s)

unique_tau_values = sorted(all_tau_values)

# ------------------ Summaries ------------------
summary = {
    model: {k: summarize(v) for k, v in stats[model].items()}
    for model in ['GPmodel', 'GPArealModel']
}
summary['VIGP_unlinked'] = {
    tau: {
        'beta':    {'mean': np.nanmean(v['beta']),    'sd': np.nanmean(v['sd_beta'])},
        'sigmasq': {'mean': np.nanmean(v['sigmasq']), 'sd': np.nanmean(v['sd_sigmasq'])},
        'tausq':   {'mean': np.nanmean(v['tausq']),   'sd': np.nanmean(v['sd_tausq'])},
        'phi':     {'mean': np.nanmean(v['phi']),     'sd': np.nanstd(v['phi'])},
        'n_correct_perm_x': {'mean': np.nanmean(v['n_correct_perm_x']), 'sd': np.nanstd(v['n_correct_perm_x'])},
        'n_correct_perm_s': {'mean': np.nanmean(v['n_correct_perm_s']), 'sd': np.nanstd(v['n_correct_perm_s'])},
    } for tau, v in stats['VIGP_unlinked'].items()
}

# ------------------ Tables ------------------
data_extended = {
    'GPmodel': [
        f"{summary['GPmodel']['beta']['mean']:.4f} ({summary['GPmodel']['beta']['sd']:.4f})",
        f"{summary['GPmodel']['phi']['mean']:.4f} ({summary['GPmodel']['phi']['sd']:.4f})",
        f"{summary['GPmodel']['tausq']['mean']:.4f} ({summary['GPmodel']['tausq']['sd']:.4f})",
        f"{summary['GPmodel']['sigmasq']['mean']:.4f} ({summary['GPmodel']['sigmasq']['sd']:.4f})",
        f"{summary['GPmodel']['n_correct_perm_x']['mean']:.4f} ({summary['GPmodel']['n_correct_perm_x']['sd']:.4f})",
        f"{summary['GPmodel']['n_correct_perm_s']['mean']:.4f} ({summary['GPmodel']['n_correct_perm_s']['sd']:.4f})",
    ],
    'GPArealModel': [
        f"{summary['GPArealModel']['beta']['mean']:.4f} ({summary['GPArealModel']['beta']['sd']:.4f})",
        f"{summary['GPArealModel']['phi']['mean']:.4f} ({summary['GPArealModel']['phi']['sd']:.4f})",
        f"{summary['GPArealModel']['tausq']['mean']:.4f} ({summary['GPArealModel']['tausq']['sd']:.4f})",
        f"{summary['GPArealModel']['sigmasq']['mean']:.4f} ({summary['GPArealModel']['sigmasq']['sd']:.4f})",
        f"{summary['GPArealModel']['n_correct_perm_x']['mean']:.4f} ({summary['GPArealModel']['n_correct_perm_x']['sd']:.4f})",
        f"{summary['GPArealModel']['n_correct_perm_s']['mean']:.4f} ({summary['GPArealModel']['n_correct_perm_s']['sd']:.4f})",
    ],
}

for tau in unique_tau_values:
    ve = summary['VIGP_unlinked'][tau]
    data_extended[f'VI_tau_{tau}'] = [
        f"{ve['beta']['mean']:.4f} ({ve['beta']['sd']:.4f})",
        f"{ve['phi']['mean']:.4f} ({ve['phi']['sd']:.4f})",
        f"{ve['tausq']['mean']:.4f} ({ve['tausq']['sd']:.4f})",
        f"{ve['sigmasq']['mean']:.4f} ({ve['sigmasq']['sd']:.4f})",
        f"{ve['n_correct_perm_x']['mean']:.4f} ({ve['n_correct_perm_x']['sd']:.4f})",
        f"{ve['n_correct_perm_s']['mean']:.4f} ({ve['n_correct_perm_s']['sd']:.4f})",
    ]

table_extended = pd.DataFrame(
    data_extended,
    index=['Beta', 'Phi', 'Tausq', 'Sigmasq', 'n_correct_perm_x', 'n_correct_perm_s']
)

print(f"B = {B}, n_i = {n_i}, folder = {phi_snr_dir}")
print(table_extended)


B = 144, n_i = 6, folder = phi_2.0_snr_1.0e+00
                          GPmodel     GPArealModel       VI_tau_0.2  \
Beta              7.8930 (0.1890)  7.9271 (0.4928)  8.0690 (0.1158)   
Phi               1.9670 (0.2104)  1.8234 (0.3966)  0.7531 (0.0087)   
Tausq             0.6169 (0.1915)  2.4764 (0.8422)  3.8109 (0.0344)   
Sigmasq           4.6806 (0.3802)  3.8967 (0.5091)  1.7432 (0.0088)   
n_correct_perm_x        nan (nan)        nan (nan)  6.0000 (0.0000)   
n_correct_perm_s        nan (nan)        nan (nan)  5.9787 (0.2052)   

                       VI_tau_0.4       VI_tau_0.6       VI_tau_0.8  \
Beta              7.9305 (0.0937)  7.5446 (0.0993)  7.2124 (0.1081)   
Phi               0.8470 (0.1311)  1.6641 (0.0119)  1.6584 (0.0102)   
Tausq             2.4962 (0.0146)  2.8480 (0.0189)  3.4328 (0.0275)   
Sigmasq           4.1342 (0.0410)  2.9973 (0.0214)  2.7261 (0.0177)   
n_correct_perm_x  6.0000 (0.0000)  6.0000 (0.0000)  6.0000 (0.0000)   
n_correct_perm_s  6.0000 (0.0

/var/folders/kj/tn21r9y15yz4_21vggchc7sc0000gp/T/ipykernel_99470/3566513806.py:65: RuntimeWarning: Mean of empty slice
  return {'mean': np.nanmean(values), 'sd': np.nanstd(values)}
/Users/debangan/Library/Python/3.12/lib/python/site-packages/numpy/lib/_nanfunctions_impl.py:2015: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
